# Does de-censoring hurt Slovene skills more? — RQ3 capability control (demo)

This notebook demonstrates the **RQ3 capability-control experiment** comparing **GaMS3-12B-Instruct** (a Slovene continued-pretraining
of Gemma 3) with **Gemma-3-12B-IT**, both run in NF4 on the same code path.

**Question:** when an *English-objective* Heretic abliteration edit (refusal-direction removal) is applied, does it damage the model's
**Slovene** capabilities more than its **English** ones, and more so in GaMS than in Gemma?

**Conditions** (per model): `orig` (unedited), `E_iter1` (iter-1 Heretic edit, λ = 1), `E_art2` (artifact 2's B' selection),
and `rand_nm_j*`, norm-matched **random-direction** edits built with Heretic's own `abliterate()` and identical parameters. Only the
directions are random, so these give the noise floor of "any edit of this size".

**Primary metric:** the headroom-normalised change `H = (acc_cond − chance)/(acc_orig − chance) − 1`, macro-averaged over tasks with
headroom ≥ 0.10, with a paired bootstrap. From it the analysis derives
`A = H_SL − H_EN` (a positive value means Slovene is hurt *less*) and the model × language interaction `I = A_GaMS − A_Gemma`.

**What the full run found:** no edit is catastrophic. A > 0 in all 4 model × edit cells. I = +0.010 [−0.065, 0.074] (E_iter1) and
−0.022 [−0.116, 0.043] (E_art2). Slovene is **not** hurt more.

### What this demo runs, and what it does not
The original `method.py` is an **orchestrator**. It calls GPU scripts that load two 12B models, build the edits, and run lm-eval,
KL, generation, Belebele and BPB, which takes about 3.5 GPU-hours. That cannot run in a 10-minute Colab session. So this notebook:
1. keeps `method.py`'s orchestration code as-is and runs it in **dry-run mode**, which prints the exact pipeline commands without launching them;
2. then runs the actual **statistical analysis step** (`src/analyze.py`: the utility block, bootstrap, macros, A, I, random-adjusted excess
   and gates, with the code copied unchanged) on **saved per-item lm-eval correctness** for a curated subset of 8 EN/SL row-paired items × 6 tasks (96 rows);
3. compares the demo-subset estimates with the full-run (300 pairs/task) reference numbers stored in the data file.

With only 8 pairs per task, the demo estimates are **very noisy**, and some tasks drop below the 0.10 headroom floor. The notebook
illustrates the method and does not replicate the result. The full-run numbers are printed next to it.

## Setup: install dependencies

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT on Colab, always install
_pip('loguru==0.7.3')

# numpy, scipy, pandas, matplotlib — pre-installed on Colab, install locally only (Colab's exact versions)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'pandas==2.2.2', 'matplotlib==3.10.0')

## Imports
The first block is `method.py`'s import block, unchanged. The second is the import block of `src/analyze.py` (the analysis step this demo executes). The last lines add pandas and matplotlib for the results cell.

In [ ]:
from __future__ import annotations

# ---- method.py imports (original) ----
import argparse
import subprocess
import sys
import time
from pathlib import Path

from loguru import logger

# ---- src/analyze.py imports (original) ----
import json
import math
import re
import zlib
from collections import defaultdict

import numpy as np
from scipy.stats import binomtest

# ---- added for the notebook's summary / visualisation ----
import pandas as pd
import matplotlib.pyplot as plt

## Data loading
`mini_demo_data.json` holds 96 rows (8 row-paired EN/SL items × 6 tasks). Each row carries the per-condition 0/1 correctness under the task's primary lm-eval metric (`acc_norm` for ARC-C/HellaSwag/OBQA/PIQA, `acc` for BoolQ/Winogrande) for both models, plus the full-run reference results.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-2cf2a7-does-slovene-taught-refusal-survive/fork/run_UESxYRggGt7E/round-3/experiment-12/demo/mini_demo_data.json"
import json
from pathlib import Path

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    local = Path("mini_demo_data.json")
    if local.exists(): return json.loads(local.read_text())
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(data["metadata"]["subset"])
print("rows:", len(data["examples"]))
print("conditions:", sorted(data["examples"][0]["correct"]))

## Configuration
Every tunable parameter is set here.
- `N_BOOT`: bootstrap replicates for all CIs. The original is **2000** (`src/common.py`).
- `DRY_RUN`: when `True`, `method.py`'s `run()` logs each pipeline command and does not launch it (the GPU steps need two 12B models and roughly 3.5 GPU-hours). Setting it to `False` only makes sense inside the full artifact repository, with its `.venv` and `src/`.
- `STEPS`, `MINUTES_PER_MODEL`, `N_GEN`: `method.py`'s CLI defaults (`--steps`, `--minutes-per-model 100`, `--n-gen 100`).
- The subset size (8 pairs per task) is fixed by `mini_demo_data.json`. The full run used 300 pairs per task.

In [ ]:
N_BOOT = 50              # original: 2000 (bootstrap replicates for every CI)
DRY_RUN = True             # original pipeline launches GPU scripts; demo only prints the commands
STEPS = "prep,mt,protocol,gams,gemma,analyze"   # original --steps default
MINUTES_PER_MODEL = 100.0  # original --minutes-per-model default
N_GEN = 100                # original --n-gen default

## Step 0: the `method.py` orchestrator (dry run)
This is `method.py`, the artifact's single entry point. Each step is a separate resumable script under `src/`:

| step | script | what it does |
|---|---|---|
| 1 | `prep_data.py` | row-paired EN/SL utility items, Belebele EN/SL/HU, Dolly harmless set, FLORES passages, overlap audit |
| 1c | `prep_mt.py` | NLLB-200 MT of the harmless set (SL-MT, EN back-translation, HU-MT) plus a chrF gate |
| 2 | `write_protocol.py` | **protocol freeze** (sha256 + git commit) before any edited-condition forward pass |
| 3-4 | `gpu_block.py` | per model: build the edits and run the manipulation check, lm-eval utility, KL, generations, Belebele, BPB, second scorer, chat-template check |
| 6-10 | `analyze.py`, `audit.py`, `assemble_p0.py`, `build_outputs.py`, `make_figs.py` | statistics, independent re-computation audit, output tables, figures |

The code is unchanged except for three notebook adaptations: (1) `WS` is the current directory, because `__file__` does not exist in a notebook;
(2) `run()` returns early when `DRY_RUN` is set; (3) `parse_args([])` uses the config values as defaults instead of reading the notebook's argv.

In [ ]:
WS = Path.cwd()  # notebook: was Path(__file__).resolve().parent
PY = str(WS / ".venv" / "bin" / "python")
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")
logger.add(str(WS / "logs" / "method.log"), rotation="30 MB", level="DEBUG")

GPU_STAGES = "util,kl,gen,belebele,bpb,scorer2,chat"


def run(args: list[str]) -> None:
    logger.info("RUN " + " ".join(args))
    if DRY_RUN:  # notebook: GPU pipeline is not launched in the demo
        return
    t = time.time()
    r = subprocess.run([PY] + args, cwd=WS)
    logger.info(f"exit {r.returncode} after {time.time() - t:.0f}s")
    if r.returncode != 0:
        raise RuntimeError(f"step failed: {args}")


@logger.catch(reraise=True)
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--steps", default=STEPS)
    ap.add_argument("--minutes-per-model", type=float, default=MINUTES_PER_MODEL)
    ap.add_argument("--n-gen", type=int, default=N_GEN)
    a = ap.parse_args([])  # notebook: ignore the kernel's argv
    steps = a.steps.split(",")
    if "prep" in steps:
        run(["src/prep_data.py"])
    if "mt" in steps:
        run(["src/prep_mt.py"])
    if "protocol" in steps:
        run(["src/write_protocol.py"])
    for m, key in (("gams3_it", "gams"), ("gemma_it", "gemma")):  # GaMS first (artifact 2's Gemma selection gets more time)
        if key in steps:
            dl = int(time.time() + a.minutes_per_model * 60)
            run(["src/gpu_block.py", "--model", m, "--stages", "pilot," + GPU_STAGES, "--n-gen", str(a.n_gen),
                 "--n-gen-lambda", "50", "--util-bs", "8",
                 "--lambda-conds", "orig,E_iter1,E_iter1_l2.0,E_iter1_l1.5,E_iter1_l0.5", "--deadline-epoch", str(dl)])
    if "analyze" in steps:
        for s in ("src/analyze.py", "src/audit.py", "src/assemble_p0.py", "src/build_outputs.py", "src/make_figs.py"):
            run([s])


main()

## Step 6a: shared constants and helpers (`src/common.py`)
These are the constants and the two decision helpers the analysis uses, copied unchanged. `headroom_H` is the primary effect size.
`gate_verdict` labels a condition CATASTROPHIC when the point loss exceeds the threshold, and POSSIBLY_CATASTROPHIC when only the 95 % upper bound does.
File-system paths from `common.py` are left out, because the data comes from `data`. `N_BOOT` comes from the config cell.

In [ ]:
SEED = 20260925

MODELS = {
    "gams3_it": {"repo": "cjvt/GaMS3-12B-Instruct", "revision": "1d0b27af5748784482600d24779409e7e1dc9adc"},
    "gemma_it": {"repo": "google/gemma-3-12b-it", "revision": "96b6f1eccf38110c56df3a15bffe176da04bfd80"},
}

UTIL_TASKS = ["arc_challenge", "boolq", "hellaswag", "openbookqa", "piqa", "winogrande"]
# lm-eval convention: acc_norm for ARC-C/HellaSwag/OBQA/PIQA; acc for BoolQ/Winogrande/Belebele
PRIMARY_METRIC = {"arc_challenge": "acc_norm", "hellaswag": "acc_norm", "openbookqa": "acc_norm", "piqa": "acc_norm",
                  "boolq": "acc", "winogrande": "acc", "belebele": "acc"}
LANGS_UTIL = ["en", "sl"]

HEADROOM_FLOOR = 0.10
GATE_LOSS = 0.20
# N_BOOT is set in the config cell (original: 2000)


def headroom_H(acc_cond: float, acc_orig: float, chance: float) -> float:
    """Headroom-normalised delta: (acc_cond - chance)/(acc_orig - chance) - 1. Loss = -H."""
    den = acc_orig - chance
    if abs(den) < 1e-12:
        return float("nan")
    return (acc_cond - chance) / den - 1.0


def gate_verdict(loss_point: float, loss_hi95: float, threshold: float = GATE_LOSS) -> str:
    """CATASTROPHIC if point loss > threshold; POSSIBLY_CATASTROPHIC if only the 95% upper bound exceeds it."""
    if loss_point is None or (isinstance(loss_point, float) and math.isnan(loss_point)):
        return "PENDING"
    if loss_point > threshold:
        return "CATASTROPHIC"
    if loss_hi95 > threshold:
        return "POSSIBLY_CATASTROPHIC"
    return "OK"

## Step 6b: load the per-item correctness and build aligned vectors (`src/analyze.py`)
In the original, `load_util` reads `results/items/util_*.jsonl`, one row per model × condition × item as written by lm-eval.
Here it builds the **same nested structure** (`U[model][cond][(lang, task)][item_id] = (correct, 1/n_choices, pair_id)`) from the rows of `data`.
`item_order` also uses the data file's row order instead of `data/util/*.jsonl`. `pct`, `ci`, `build_vectors` and `boot_indices` are unchanged.

`build_vectors` keeps only items scored under `orig` **and** every condition. EN and SL share the same `pair_id` order, so a single bootstrap
resample of pair indices applies to both languages and both models. That is the **joint paired bootstrap** behind the A and I CIs.

In [ ]:
RNG_SEED = SEED + 1
MODEL_LIST = list(MODELS)


def pct(a: np.ndarray, q: float) -> float:
    return float(np.percentile(a, q))


def ci(a: np.ndarray, level: int = 95) -> list[float]:
    lo = (100 - level) / 2
    return [round(pct(a, lo), 5), round(pct(a, 100 - lo), 5)]


def load_util(kind: str) -> dict:
    """-> U[model][cond][lang][task] = {item_id: correct(primary metric)}"""
    U: dict = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
    for r in data["examples"]:  # notebook: was results/items/{kind}_*.jsonl (one row per model x cond x item)
        for mc, correct in r["correct"].items():
            model, cond = mc.split("__")
            U[model][cond][(r["lang"], r["task"])][r["item_id"]] = (float(correct), 1.0 / r["n_choices"],
                                                                     r.get("pair_id"))
    return U


def item_order(task: str, lang: str) -> list[str]:
    # notebook: was the row order of data/util/{task}_{lang}.jsonl
    return [r["item_id"] for r in data["examples"] if r["task"] == task and r["lang"] == lang]


def build_vectors(U: dict, model: str, conds: list[str], cells: list[tuple[str, str]]) -> dict:
    """Aligned vectors over the item ids present in orig AND every cond (intersection), per (lang, task) cell.
    Paired tasks: EN/SL share pair_id -> the SAME pair order is used in both languages."""
    out = {}
    for lang, task in cells:
        if (lang, task) not in U[model].get("orig", {}):
            continue
        common = None
        for c in ["orig"] + conds:
            ids = set(U[model].get(c, {}).get((lang, task), {}).keys())
            common = ids if common is None else common & ids
        order = [i for i in item_order(task, lang) if i in common]
        if not order:
            continue
        vec = {c: np.array([U[model][c][(lang, task)][i][0] for i in order]) for c in ["orig"] + conds}
        chance = float(np.mean([U[model]["orig"][(lang, task)][i][1] for i in order]))
        out[(lang, task)] = {"ids": order, "vec": vec, "chance": chance}
    return out


def boot_indices(n: int, rng: np.random.Generator, B: int = None) -> np.ndarray:
    return rng.integers(0, n, size=(B if B is not None else N_BOOT, n))  # notebook: default B = config N_BOOT

## Step 6c: the utility block (`util_block`, unchanged)
For every model × condition × language × task cell this computes:
- accuracy of `orig` and of the condition, chance, headroom, **H** with a bootstrap CI, raw percentage-point change, and a McNemar exact test (Holm-corrected within model × condition);
- per-language **macro H** over *eligible* tasks (headroom ≥ 0.10), with 90/95 % CIs, and the macro loss upper bound used by the gate;
- **A = H_SL − H_EN** and **I = A_GaMS − A_Gemma**, on the same bootstrap indices;
- **random-adjusted excess**: H(edit) − H(norm-matched random edit `rand_nm_j1`), which separates edit-specific damage from the generic cost of perturbing the weights.

`np.errstate` suppresses divide-by-zero warnings for bootstrap resamples where `acc_orig == chance`. Non-finite replicates are dropped before the percentiles.

In [ ]:
def util_block(U: dict, conds_all: list[str], langs: list[str], tasks: list[str], label: str, rng_seed: int) -> dict:
    """Compute H per cell, macros, A, I, H_excess with a joint paired bootstrap (same resampled pair indices across
    conditions, languages and models within a task)."""
    rng = np.random.default_rng(rng_seed)
    cells = [(l, t) for l in langs for t in tasks]
    V = {m: build_vectors(U, m, [c for c in conds_all if c in U[m]], cells) for m in MODEL_LIST if m in U}
    # shared bootstrap indices per task: need the same n across models/languages -> use pair-index intersection
    idx = {}
    for t in tasks:
        ns = [len(V[m][(l, t)]["ids"]) for m in V for l in langs if (l, t) in V[m]]
        if not ns:
            continue
        if len(set(ns)) != 1:
            logger.warning(f"{label}/{t}: unequal n across cells {ns}; bootstrap per-cell independent for this task")
            idx[t] = None
        else:
            idx[t] = boot_indices(ns[0], rng)
    table: dict = {}
    boots: dict = {}
    for m in V:
        conds = [c for c in conds_all if c in U[m] and c != "orig"]
        table[m] = {}
        for c in conds:
            table[m][c] = {}
            for l in langs:
                table[m][c][l] = {}
                for t in tasks:
                    cell = V[m].get((l, t))
                    if cell is None or c not in cell["vec"]:
                        continue
                    vo, vc, ch = cell["vec"]["orig"], cell["vec"][c], cell["chance"]
                    ao, ac = float(vo.mean()), float(vc.mean())
                    H = headroom_H(ac, ao, ch)
                    ix = idx.get(t) if idx.get(t) is not None else boot_indices(len(vo), rng)
                    bo, bc = vo[ix].mean(1), vc[ix].mean(1)
                    with np.errstate(divide="ignore", invalid="ignore"):
                        bH = (bc - ch) / (bo - ch) - 1
                    b_ = int(((vo == 1) & (vc == 0)).sum())
                    c_ = int(((vo == 0) & (vc == 1)).sum())
                    p_mc = float(binomtest(min(b_, c_), b_ + c_, 0.5).pvalue) if b_ + c_ > 0 else 1.0
                    table[m][c][l][t] = {"acc_orig": round(ao, 5), "acc": round(ac, 5), "acc_metric": PRIMARY_METRIC[t],
                                         "chance": round(ch, 5), "headroom": round(ao - ch, 5), "H": round(H, 5),
                                         "H_ci95": ci(bH[np.isfinite(bH)]), "raw_pp": round(100 * (ac - ao), 3),
                                         "rel_pct": round(100 * (ac / ao - 1), 3) if ao > 0 else None, "n": len(vo),
                                         "eligible": bool(ao - ch >= HEADROOM_FLOOR - 1e-9), "mcnemar_b_orig_right_cond_wrong": b_,
                                         "mcnemar_c_orig_wrong_cond_right": c_, "mcnemar_p": p_mc}
                    boots[(m, c, l, t)] = (bH, (bc - bo) * 100)
                # Holm within condition happens after all langs
            # Holm-correct McNemar within model x condition
            ps = [(l, t, table[m][c][l][t]["mcnemar_p"]) for l in table[m][c] for t in table[m][c][l]]
            order = sorted(range(len(ps)), key=lambda k: ps[k][2])
            running = 0.0
            for rank, k in enumerate(order):
                adj = min(1.0, (len(ps) - rank) * ps[k][2])
                running = max(running, adj)
                l, t, _ = ps[k]
                table[m][c][l][t]["mcnemar_p_holm"] = round(running, 6)
    # macros
    macros: dict = {}
    macro_boot: dict = {}
    RAW_H: dict = {}
    for m in table:
        macros[m] = {}
        for c in table[m]:
            macros[m][c] = {}
            for l in langs:
                cells_l = table[m][c].get(l, {})
                if not cells_l:
                    continue
                elig = [t for t in cells_l if cells_l[t]["eligible"]]
                inel = [t for t in cells_l if not cells_l[t]["eligible"]]
                Hm = float(np.mean([cells_l[t]["H"] for t in elig])) if elig else float("nan")
                bHm = np.mean([boots[(m, c, l, t)][0] for t in elig], axis=0) if elig else np.full(N_BOOT, np.nan)
                raw = float(np.mean([cells_l[t]["raw_pp"] for t in cells_l]))
                braw = np.mean([boots[(m, c, l, t)][1] for t in cells_l], axis=0)
                rel = float(np.mean([cells_l[t]["rel_pct"] for t in cells_l if cells_l[t]["rel_pct"] is not None]))
                macro_boot[(m, c, l)] = (bHm, braw)
                RAW_H[(m, c, l)] = Hm
                macros[m][c][l] = {"macro_H": round(Hm, 5), "macro_H_ci95": ci(bHm[np.isfinite(bHm)]) if elig else None,
                                   "macro_H_ci90": ci(bHm[np.isfinite(bHm)], 90) if elig else None,
                                   "macro_loss": round(-Hm, 5), "macro_loss_hi95": round(-pct(bHm[np.isfinite(bHm)], 2.5), 5) if elig else None,
                                   "eligible_tasks": elig, "ineligible_tasks": inel, "raw_pp_macro": round(raw, 4),
                                   "raw_pp_macro_ci95": ci(braw), "rel_pct_macro": round(rel, 4),
                                   "F6_raw_pp_rule": len(inel) >= 3}
            if all(l in macros[m][c] for l in ("en", "sl")) and len(langs) >= 2:
                pooled = (macro_boot[(m, c, "en")][0] + macro_boot[(m, c, "sl")][0]) / 2
                A = macro_boot[(m, c, "sl")][0] - macro_boot[(m, c, "en")][0]
                macros[m][c]["pooled_macro_H"] = round((RAW_H[(m, c, "en")] + RAW_H[(m, c, "sl")]) / 2, 5)
                macros[m][c]["pooled_macro_H_ci95"] = ci(pooled[np.isfinite(pooled)])
                macros[m][c]["A_sl_minus_en"] = round(RAW_H[(m, c, "sl")] - RAW_H[(m, c, "en")], 5)
                macros[m][c]["A_ci95"] = ci(A[np.isfinite(A)])
                macros[m][c]["A_ci90"] = ci(A[np.isfinite(A)], 90)
    # interaction and random-adjusted
    inter: dict = {}
    if len(macros) == 2:
        mg, mm = "gams3_it", "gemma_it"
        for c in set(macros.get(mg, {})) & set(macros.get(mm, {})):
            if "A_sl_minus_en" in macros[mg][c] and "A_sl_minus_en" in macros[mm][c]:
                bA = {m: macro_boot[(m, c, "sl")][0] - macro_boot[(m, c, "en")][0] for m in (mg, mm)}
                bI = bA[mg] - bA[mm]
                inter[c] = {"I": round((RAW_H[(mg, c, "sl")] - RAW_H[(mg, c, "en")]) - (RAW_H[(mm, c, "sl")] - RAW_H[(mm, c, "en")]), 5),
                            "I_ci95": ci(bI[np.isfinite(bI)]), "I_ci90": ci(bI[np.isfinite(bI)], 90),
                            "note": "resolution limit ~0.10-0.12 of headroom (80% power, alpha .05); |I| below it is an estimate, not 'no asymmetry'"}
    excess: dict = {}
    for m in macros:
        for c in macros[m]:
            if c.startswith("E_") and "rand_nm_j1" in macros[m]:
                excess.setdefault(m, {})[c] = {}
                for l in langs:
                    if l in macros[m][c] and l in macros[m]["rand_nm_j1"]:
                        b = macro_boot[(m, c, l)][0] - macro_boot[(m, "rand_nm_j1", l)][0]
                        excess[m][c][l] = {"H_excess": round(RAW_H[(m, c, l)] - RAW_H[(m, "rand_nm_j1", l)], 5),
                                           "ci95": ci(b[np.isfinite(b)])}
                if "en" in excess[m][c] and "sl" in excess[m][c]:
                    bA = (macro_boot[(m, c, "sl")][0] - macro_boot[(m, c, "en")][0]) - \
                         (macro_boot[(m, "rand_nm_j1", "sl")][0] - macro_boot[(m, "rand_nm_j1", "en")][0])
                    excess[m][c]["A_excess"] = {"value": round((RAW_H[(m, c, "sl")] - RAW_H[(m, "rand_nm_j1", "sl")]) - (RAW_H[(m, c, "en")] - RAW_H[(m, "rand_nm_j1", "en")]), 5),
                                                "ci95": ci(bA[np.isfinite(bA)])}
    return {"cells": table, "macros": macros, "interaction": inter, "random_adjusted": excess,
            "_macro_boot": macro_boot}

## Step 6d: capability gate (`gates_from`, unchanged)
The gate asks whether each edit is **catastrophic** for utility. By default it uses the headroom macro loss against the 0.20 threshold.
When 3 or more tasks in a language fall below the headroom floor, it switches to the raw-pp rule against a 5 pp threshold (the "F6" rule).
The worst language decides the verdict. The full run also has a λ-subset block (λ = 0.5/1.5/2.0 on 100 items × 4 tasks). That block is not in the demo data, so `lam=None` here.

**Caution:** with only 8 pairs per task, many cells fall below the headroom floor, and the bootstrap upper bounds are much wider than in the full run.
The demo gate verdicts are therefore not comparable to the full-run verdicts, which are printed next to them.

In [ ]:
def gates_from(util: dict, lam: dict | None) -> dict:
    g: dict = {}
    for src, blk in (("full", util), ("lambda_subset", lam)):
        if not blk:
            continue
        for m, cc in blk["macros"].items():
            for c, d in cc.items():
                langs = [l for l in ("en", "sl") if l in d]
                verdicts = {}
                for l in langs:
                    x = d[l]
                    if x["F6_raw_pp_rule"]:
                        loss = -x["raw_pp_macro"]
                        hi = -x["raw_pp_macro_ci95"][0]
                        verdicts[l] = {"rule": "F6 raw-pp (5 pp)", "loss": loss, "loss_hi95": hi, "verdict": gate_verdict(loss, hi, 5.0)}
                    else:
                        verdicts[l] = {"rule": "headroom macro (0.20)", "loss": x["macro_loss"], "loss_hi95": x["macro_loss_hi95"],
                                       "verdict": gate_verdict(x["macro_loss"], x["macro_loss_hi95"] if x["macro_loss_hi95"] is not None else float("nan"))}
                    # co-reported sensitivity (decides nothing on its own): raw-pp macro loss vs the F6 5-pp threshold
                    verdicts[l]["sensitivity_raw_pp"] = {"loss_pp": -x["raw_pp_macro"], "loss_pp_hi95": -x["raw_pp_macro_ci95"][0],
                                                         "verdict_5pp": gate_verdict(-x["raw_pp_macro"], -x["raw_pp_macro_ci95"][0], 5.0)}
                rank = {"OK": 0, "POSSIBLY_CATASTROPHIC": 1, "CATASTROPHIC": 2, "PENDING": -1}
                worst = max((v["verdict"] for v in verdicts.values()), key=lambda s: rank[s]) if verdicts else "PENDING"
                if c in g.get(m, {}) and g[m][c]["source"] == "full":
                    g[m][c]["lambda_subset_check"] = {"per_lang": verdicts, "verdict": worst}  # full-set gate takes precedence
                    continue
                g.setdefault(m, {})[c] = {"source": src, "per_lang": verdicts, "verdict": worst}
    return g

## Step 6e: run the utility analysis (the first part of `analyze.main()`)
These are the same calls as the start of `analyze.main()`: load the utility items, list the edited conditions, run `util_block` over EN/SL × 6 tasks, and derive the gates.
The rest of `main()` (λ subset, Belebele, chat sensitivity, KL, BPB, generations, second scorer) needs data that is not in this demo subset.

In [ ]:
t0 = time.time()
U = load_util("util")
conds_full = sorted({c for m in U for c in U[m] if c != "orig"})
util = util_block(U, ["orig"] + conds_full, ["en", "sl"], UTIL_TASKS, "util", RNG_SEED)
logger.info(f"utility conds: {conds_full}")
gates = gates_from(util, None)
for m in gates:
    for c, g in gates[m].items():
        logger.info(f"GATE {m}/{c}: {g['verdict']} {json.dumps({l: (round(v['loss'], 4), v['verdict']) for l, v in g['per_lang'].items()})}")
print(f"analysis runtime: {time.time() - t0:.1f}s  (N_BOOT={N_BOOT})")

## Results: demo subset vs full run
The table puts the demo-subset estimates (8 pairs per task) next to the full-run reference (300 pairs per task, from `full_method_out.json`) for
macro H in EN and SL, **A = H_SL − H_EN**, and the gate verdict. The second table shows the interaction **I = A_GaMS − A_Gemma**.
The figure plots A per model × condition. Filled markers are the demo subset and hollow markers are the full run, each with a 95 % bootstrap CI.

How to read it: in the full run, A > 0 for every real edit (E_iter1, E_art2) in both models, which means Slovene is *not* hurt more than English.
I's CI straddles 0, so GaMS is not differentially more fragile in Slovene. The demo subset is far too small to resolve these effects.
Its CIs are expected to be several times wider.

In [ ]:
REF = data["metadata"]["full_run_reference"]


def _fmt(v):
    return "nan" if v is None or (isinstance(v, float) and math.isnan(v)) else f"{v:+.3f}"


rows = []
for m in MODEL_LIST:
    for c in sorted(util["macros"].get(m, {})):
        d = util["macros"][m][c]
        r = REF["macros"].get(m, {}).get(c, {})
        rows.append({"model": m, "cond": c,
                     "H_EN demo": _fmt(d.get("en", {}).get("macro_H")), "H_EN full": _fmt(r.get("en", {}).get("macro_H")),
                     "H_SL demo": _fmt(d.get("sl", {}).get("macro_H")), "H_SL full": _fmt(r.get("sl", {}).get("macro_H")),
                     "A demo": _fmt(d.get("A_sl_minus_en")), "A demo CI95": d.get("A_ci95"),
                     "A full": _fmt(r.get("A_sl_minus_en")), "A full CI95": r.get("A_ci95"),
                     "gate demo": gates.get(m, {}).get(c, {}).get("verdict"), "gate full": REF["gates"].get(m, {}).get(c),
                     "eligible EN/SL (demo)": f"{len(d.get('en', {}).get('eligible_tasks', []))}/{len(d.get('sl', {}).get('eligible_tasks', []))}"})
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
print("Macro headroom-normalised change H, A = H_SL - H_EN, and gate (demo subset vs full run)")
print(pd.DataFrame(rows).to_string(index=False))

irows = [{"cond": c, "I demo": _fmt(util["interaction"].get(c, {}).get("I")), "I demo CI95": util["interaction"].get(c, {}).get("I_ci95"),
          "I full": _fmt(REF["interaction"].get(c, {}).get("I")), "I full CI95": REF["interaction"].get(c, {}).get("I_ci95")}
         for c in sorted(set(util["interaction"]) | set(REF["interaction"]))]
print("\nModel x language interaction I = A_GaMS - A_Gemma")
print(pd.DataFrame(irows).to_string(index=False))

# per-task raw percentage-point change for E_iter1 (demo subset)
trows = []
for m in MODEL_LIST:
    for l in ("en", "sl"):
        for t, cell in util["cells"].get(m, {}).get("E_iter1", {}).get(l, {}).items():
            trows.append({"model": m, "lang": l, "task": t, "acc_orig": cell["acc_orig"], "acc_E_iter1": cell["acc"],
                          "raw_pp": cell["raw_pp"], "headroom": cell["headroom"], "eligible": cell["eligible"], "n": cell["n"]})
print("\nPer-task cells for E_iter1 (demo subset)")
print(pd.DataFrame(trows).to_string(index=False))

# ---- figure: A per model x condition, demo vs full ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
ax = axes[0]
labels, k = [], 0
for m, col in (("gams3_it", "tab:blue"), ("gemma_it", "tab:orange")):
    for c in sorted(set(util["macros"].get(m, {})) | set(REF["macros"].get(m, {}))):
        for src, blk, mk, off in (("demo", util["macros"], "o", -0.15), ("full", REF["macros"], "s", 0.15)):
            d = blk.get(m, {}).get(c, {})
            if d.get("A_sl_minus_en") is None or d.get("A_ci95") is None:
                continue
            a, (lo, hi) = d["A_sl_minus_en"], d["A_ci95"]
            ax.errorbar(k + off, a, yerr=[[a - lo], [hi - a]], fmt=mk, color=col, capsize=3,
                        mfc=col if src == "demo" else "white")
        labels.append(f"{m.split('_')[0]}\n{c}")
        k += 1
ax.axhline(0, color="grey", lw=0.8, ls="--")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=7)
ax.set_ylabel("A = H_SL - H_EN  (>0: SL hurt less)")
ax.set_title("Language asymmetry A (filled: demo subset, hollow: full run)")

ax = axes[1]
cs = sorted(set(util["interaction"]) | set(REF["interaction"]))
for j, c in enumerate(cs):
    for blk, mk, off, mfc in ((util["interaction"], "o", -0.12, "tab:green"), (REF["interaction"], "s", 0.12, "white")):
        d = blk.get(c)
        if not d:
            continue
        v, (lo, hi) = d["I"], d["I_ci95"]
        ax.errorbar(j + off, v, yerr=[[v - lo], [hi - v]], fmt=mk, color="tab:green", mfc=mfc, capsize=3)
ax.axhline(0, color="grey", lw=0.8, ls="--")
ax.set_xticks(range(len(cs)))
ax.set_xticklabels(cs)
ax.set_ylabel("I = A_GaMS - A_Gemma")
ax.set_title("Model x language interaction (filled: demo, hollow: full)")
plt.tight_layout()
plt.show()